## Imports

In [1]:
import pandas as pd
import zipfile as zipfile
import pathlib
import os
import shutil
import requests

## Functions

In [2]:
def download_data_mdl(filepath, url):
    """ This function download file from a specified url
        Argument:
        - url : a url
    """

    response = requests.get(url) 

    with open(filepath, mode='wb') as f: # mode wb : write + byte
        f.write(response.content)

    with zipfile.ZipFile(filepath, 'r') as data_zip:
        print(data_zip.namelist())

    return


def extract_zip(filepath='data_mdl/ml-20m.zip'):
    """ This function extract 1 zip file
        Argument:
        filepath = path of the zip file to extract
    """
    # Extraction of directory name from filepath
    dir = os.path.dirname(filepath)
    
    # Decompression of the zip file in the same directory
    with zipfile.ZipFile(filepath, mode="r") as archive:
        archive.extractall(dir)
    
    return



def split_file_in_two(filepath='data_mdl/ml-20m/ratings.csv'):
    """ This function divide a file in 2 parts and suppress original file
        Argument:
            file : path of the file to split
    """

    # Extraction of existing file path elements
    dir            = os.path.dirname(filepath)
    file_name      = os.path.basename(filepath).split('.', 1)[0]
    file_extension = os.path.basename(filepath).split('.', 1)[1]

    # Build of new files path:
    # Exemple : 'data/ml-20m/ratings.csv' => 'data/ml-20m/ratings_part1.csv' and 'data/ml-20m/ratings_part2.csv'
    part1 = os.path.join(dir,file_name + '_part1' + '.'+file_extension)
    part2 = os.path.join(dir,file_name + '_part2' + '.'+file_extension)

    # Load of file in Pandas
    df_to_split = pd.read_csv(filepath, sep=',')

    # Define the middle of file to split
    limit = df_to_split.shape[0] // 2

    # Split and save using new path files
    df_to_split[:limit].to_csv(part1, sep=',')
    df_to_split[limit:].to_csv(part2, sep=',')

    # Suppression of the original file
    os.remove(filepath)

    return


def compress_files(input_dir ='data_mdl/ml-20m/', output_dir = 'data'):
    """ This function compress files stored in input_dir 
        Arguments : 
            input_dir = directory containing files to compress
            output_dir = directory where compressed files will be stored
    """

    # Creation of files lists
    input_filepaths = []
    output_filepaths = []

    for file in os.listdir(input_dir):
        input_filepaths.append(os.path.join(input_dir, file))
        output_filepaths.append(os.path.join(output_dir, file[:-4]+'.zip'))

    # Compression of listed files 
    for input,output in zip(input_filepaths, output_filepaths):
        with zipfile.ZipFile(output, mode='w', compression=zipfile.ZIP_DEFLATED) as myzip:
            myzip.write(input)

    return



def clean_datafolder(zipfilepath='data_mdl/ml-20m.zip', extraction_dir='data_mdl/ml-20m'):
    """ This function remove original sip file and the folder created during extraction
        Arguments:
            zipfilepath
            extraction_dir
    """

    os.remove(zipfilepath)        # remove a file only
    shutil.rmtree(extraction_dir) # remove a folder and the content of the folder

    return

## Data Download

In [3]:

# To get the link : in the page https://grouplens.org/datasets/movielens/20m/, right clic on the file and copy-link

download_data_mdl(filepath = 'data_mdl/ml-20m.zip', url='https://files.grouplens.org/datasets/movielens/ml-20m.zip')


['ml-20m/', 'ml-20m/genome-scores.csv', 'ml-20m/genome-tags.csv', 'ml-20m/links.csv', 'ml-20m/movies.csv', 'ml-20m/ratings.csv', 'ml-20m/README.txt', 'ml-20m/tags.csv']


## Data Reformat

In [4]:

# Step 1  : extraction of the zip file
extract_zip(filepath='data_mdl/ml-20m.zip')

# Step 2 : split of rating file that will be over GitHub limit of 100 Mo
split_file_in_two(filepath='data_mdl/ml-20m/ratings.csv')

# Step 3 : Compression of individual files in zip format
compress_files(input_dir ='data_mdl/ml-20m/', output_dir = 'data_mdl')

# Step 4 : Suppression of the original zip file and of the extraction folder
clean_datafolder(zipfilepath='data_mdl/ml-20m.zip', extraction_dir='data_mdl/ml-20m')